##### This is for the Metrics results given from the CSV evaluation 

##### This is just the overview 

In [ ]:
#importing the libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, cohen_kappa_score)

In [86]:
#loading the data

df = pd.read_csv('evaluation_results3.csv')
df.head(5)

,ID,Data Provider,Project Name,Consumer Team,Consumer Name,Consumer Description,Variation Type,Variation Value,Purpose,Realistic?,Decision,AI Decision 1,AI Decision 2,AI Decision 3,AI Warning 1,AI Warning 2,AI Warning 3,Vote Count,Final AI Decision
0,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,NaN,Original Request,"To improve our risk assessment process, we aim...","Yes, this is a realistic access request for a ...",Accept,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
1,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"intern + very hasty (typos, shorthand, missing...",hey need access to old claims data asap gotta ...,NaN,NaN,Reject,Reject,Reject,"{'title': 'Default policies', 'policyKey': 'gl...","{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'default', 'title': 'Default pol...",0-3,Reject
2,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"intern + neutral (standard professional, no pa...",I'm looking to access historical claims data t...,NaN,NaN,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
3,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,intern + very formal (precise legal-style lang...,I hereby formally request access to the histor...,NaN,NaN,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
4,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"junior analyst + very hasty (typos, shorthand,...",hey can I get access to the old claims data? n...,NaN,NaN,Reject,Reject,Reject,"{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'Default policies', 'title': 'Da...",0-3,Reject


In [87]:
#Gathering information about this data

print(df.info())
print(df.describe())
print(df.shape) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1417 entries, 0 to 1416
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ID                    1417 non-null   object
 1   Data Provider         1417 non-null   object
 2   Project Name          1417 non-null   object
 3   Consumer Team         1417 non-null   object
 4   Consumer Name         1417 non-null   object
 5   Consumer Description  1417 non-null   object
 6   Variation Type        1308 non-null   object
 7   Variation Value       1417 non-null   object
 8   Purpose               1417 non-null   object
 9   Realistic?            109 non-null    object
 10  Decision              109 non-null    object
 11  AI Decision 1         1417 non-null   object
 12  AI Decision 2         1417 non-null   object
 13  AI Decision 3         1417 non-null   object
 14  AI Warning 1          632 non-null    object
 15  AI Warning 2          633 non-null    

In [88]:
#Finding the number of the original requests and the modified ones 

df["Original Requests"] = df["Variation Value"].astype(str).str.strip().str.lower().eq("original request")


print(f"Original Requests: {df['Original Requests'].sum()}")
print(f"Modified Requests: {len(df) - df['Original Requests'].sum()}")

Original Requests: 109
Modified Requests: 1308


In [89]:
#extracting columns for the seniority and the hastiness dimension.

def extracting_seniority(text):
    text = str(text).lower()

    if "intern" in text:
        return "Intern"
    elif "junior analyst" in text:
        return "Junior Analyst"
    elif "senior manager" in text:
        return "Senior Manager"
    elif "executive/ceo" in text:
        return "Executive/CEO"
    #elif "original request" in text:
    #    return "Original"
    return None


def extracting_hastiness(text):
    text = str(text).lower()

    if "very hasty" in text:
        return "Very Hasty"
    elif "neutral" in text:
        return "Neutral"
    elif "very formal" in text:
        return "Very Formal"
    #elif "original request" in text:
        #return "Original"
    return None

df['Seniority'] = df['Variation Value'].apply(extracting_seniority)
df['Hastiness'] = df['Variation Value'].apply(extracting_hastiness)



seniority_category=["Intern", "Junior Analyst", "Senior Manager", "Executive/CEO", "Original"]
hastiness_category=["Very Hasty", "Neutral", "Very Formal", "Original"]

df["Seniority"] = pd.Categorical(df["Seniority"],categories = seniority_category, ordered=True)
df["Hastiness"] = pd.Categorical(df["Hastiness"],categories = hastiness_category, ordered=True)

pd.crosstab(df['Seniority'], df['Hastiness'], margins=True, margins_name="Total")

Hastiness,Very Hasty,Neutral,Very Formal,Total
Seniority,,,,
Intern,109,109,109,327
Junior Analyst,109,109,109,327
Senior Manager,109,109,109,327
Executive/CEO,109,109,109,327
Total,436,436,436,1308


In [90]:
# creating a dataframe only showcasing the accept/rejection rates for the original (unmodified) requests

original_requests = df[df["Original Requests"]].copy()

decision_counts = original_requests["Decision"].value_counts()
final_ai_counts = original_requests["Final AI Decision"].value_counts()

# display counts side-by-side
original_requests_info = pd.DataFrame({"Decision": decision_counts,"Final AI Decision": final_ai_counts}).fillna(0).astype(int)

display(original_requests_info)


,Decision,Final AI Decision
Accept,93,80
Reject,16,29


In [91]:
#Gathering the countd of the rates of each AI Decision

ai_decision_col = [
    "AI Decision 1",
    "AI Decision 2",
    "AI Decision 3",
    "Final AI Decision"
]
counts = df[ai_decision_col].apply(pd.Series.value_counts)
display(counts)

,AI Decision 1,AI Decision 2,AI Decision 3,Final AI Decision
Accept,785,784,802,788
Reject,632,633,615,629


In [92]:
#Analysing the voting stability between each AI Decision

voting_summary = (df["Vote Count"].value_counts().rename_axis("Vote Count [Accept - Reject]").reset_index(name="Count"))

display(voting_summary)

,Vote Count [Accept - Reject],Count
0,3-0,549
1,0-3,383
2,1-2,246
3,2-1,239


In [93]:
#making a transition table to see how many were changed from Accept to Reject and vice versa

#creating a dataframe containing only the modified requests
modified_df = df[~df["Original Requests"]].copy()

#mapping the original AI decision to every modified version with the same ID
original_lookup = original_requests.set_index("ID")["Final AI Decision"]
modified_df["Original Final AI Decision"] = modified_df["ID"].map(original_lookup)

#creating a transition label
modified_df["Transition"] = (
    modified_df["Original Final AI Decision"]
    + " -> "
    + modified_df["Final AI Decision"]
)

#counting each transition
transition_table = (
    modified_df["Transition"]
    .value_counts()
    .reindex(
        [
            "Accept -> Accept",
            "Accept -> Reject",
            "Reject -> Reject",
            "Reject -> Accept"
        ],
        fill_value=0
    )
    .rename_axis("Transition")
    .reset_index(name="Count")
)

display(transition_table)

,Transition,Count
0,Accept -> Accept,586
1,Accept -> Reject,374
2,Reject -> Reject,226
3,Reject -> Accept,122


##### The calculating metrics part now
- to figure out how well Governance AI performed for these modified requests

In [94]:
y_true = original_requests["Decision"]
y_pred = original_requests["Final AI Decision"]


accuracy = accuracy_score(y_true, y_pred) #calculating the accuracy score
recall = recall_score(y_true, y_pred, pos_label='Accept') #calculating the recall score
precision = precision_score(y_true, y_pred, pos_label='Accept') #calculating the precision score
f1 = f1_score(y_true, y_pred, pos_label='Accept') #calculating the f1 score
kappa = cohen_kappa_score(y_true, y_pred) #calculating the cohen kappa score


#making a table to display all the results of the metrics

metrics_results = pd.DataFrame({
    "Metric": ["Accuracy", "Recall", "Precision", "F1 Score", "Cohen Kappa"],
    "Score": [accuracy, recall, precision, f1, kappa]
})


display(metrics_results)


,Metric,Score
0,Accuracy,0.678899
1,Recall,0.741935
2,Precision,0.862500
3,F1 Score,0.797688
4,Cohen Kappa,0.040734


In [95]:
#confusion matrix

cm = confusion_matrix(y_true, y_pred, labels=["Accept", "Reject"])
cm

array([[69, 24],
       [11,  5]])

In [103]:
#finding the accuracy by condition

expert_lookup = original_requests.set_index("ID")["Decision"]

modified_df["Expert Decision"] = modified_df["ID"].map(expert_lookup)


accuracy_results = []

for (seniority, hastiness), group in modified_df.groupby(["Seniority", "Hastiness"],observed=True):
    accuracy = accuracy_score(group["Expert Decision"], group["Final AI Decision"])

    accuracy_results.append({
        "Seniority": seniority,
        "Hastiness": hastiness,
        "Accuracy": accuracy * 100
    })

accuracy_df = pd.DataFrame(accuracy_results)
display(accuracy_df)

print()

display(accuracy_df.pivot(index="Seniority", columns="Hastiness", values="Accuracy").round(2).fillna(0))

,Seniority,Hastiness,Accuracy
0,Intern,Very Hasty,30.275229
1,Intern,Neutral,52.293578
2,Intern,Very Formal,64.220183
3,Junior Analyst,Very Hasty,34.862385
4,Junior Analyst,Neutral,58.715596
5,Junior Analyst,Very Formal,62.385321
6,Senior Manager,Very Hasty,38.532110
7,Senior Manager,Neutral,67.889908
8,Senior Manager,Very Formal,66.055046
9,Executive/CEO,Very Hasty,38.532110


Hastiness,Neutral,Very Formal,Very Hasty
Seniority,,,
Executive/CEO,66.97,61.47,38.53
Intern,52.29,64.22,30.28
Junior Analyst,58.72,62.39,34.86
Senior Manager,67.89,66.06,38.53
